In [ ]:
https://github.com/24110207-Hai-Nguyen/8_puzzle_complex

In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox
import random
import collections
import heapq
import time
import threading
from itertools import permutations

# --- CẤU HÌNH BÀI TOÁN 8-PUZZLE ---
GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)

class Node:
    _id_counter = 65
    def __init__(self, state, parent=None, action=None, cost=0, depth=0, h_cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.cost = cost
        self.depth = depth
        self.h_cost = h_cost
        self.f_cost = cost + h_cost
        if parent is None:
            self.name = "A"
        else:
            self.name = chr(Node._id_counter)
            Node._id_counter += 1
            if Node._id_counter > 90:
                Node._id_counter = 65
    def __lt__(self, other):
        return self.f_cost < other.f_cost

def get_misplaced_tiles(state):
    count = 0
    for i in range(9):
        if state[i] != 0 and state[i] != GOAL[i]:
            count += 1
    return count

def get_value(state):
    return 9 - get_misplaced_tiles(state)

def get_neighbors(state):
    neighbors = []
    idx = state.index(0)
    row, col = divmod(idx, 3)
    moves = [(-1, 0, 'Up'), (1, 0, 'Down'), (0, -1, 'Left'), (0, 1, 'Right')]
    for dr, dc, move in moves:
        r, c = row + dr, col + dc
        if 0 <= r < 3 and 0 <= c < 3:
            new_idx = r * 3 + c
            new_state = list(state)
            new_state[idx], new_state[new_idx] = new_state[new_idx], new_state[idx]
            neighbors.append((tuple(new_state), move))
    return neighbors

def format_state_matrix(state):
    matrix_str = ""
    for i in range(3):
        row = [str(x) if x != 0 else "_" for x in state[i*3:(i+1)*3]]
        matrix_str += " " + " ".join(row) + "\n"
    return matrix_str

# --- HỖ TRỢ COMPLEX MODE ---
def count_inversions(state):
    inv = 0
    for i in range(9):
        for j in range(i+1, 9):
            if state[i] != 0 and state[j] != 0 and state[i] > state[j]:
                inv += 1
    return inv

def is_solvable(state):
    return count_inversions(state) % 2 == 0

def generate_random_solvable_states(num):
    states = []
    while len(states) < num:
        state = list(GOAL)
        for _ in range(30):
            neighbors = get_neighbors(tuple(state))
            state = list(random.choice(neighbors)[0])
        if is_solvable(state) and tuple(state) not in states:
            states.append(tuple(state))
    return states

def parse_template(template_str):
    parts = template_str.strip().split()
    if len(parts) != 9:
        return []
    template = []
    unknown_indices = []
    present_numbers = set()
    for i, p in enumerate(parts):
        if p == '?':
            template.append('?')
            unknown_indices.append(i)
        else:
            try:
                num = int(p)
                if num < 0 or num > 8 or num in present_numbers:
                    return []
                present_numbers.add(num)
                template.append(num)
            except:
                return []
    missing_numbers = [x for x in range(9) if x not in present_numbers]
    if len(missing_numbers) != len(unknown_indices):
        return []
    if len(unknown_indices) > 5:
        messagebox.showwarning("Warning", "Too many '?' (max 5) to avoid performance issues.")
        return []
    possible = []
    for perm in permutations(missing_numbers):
        state_list = template[:]
        for idx, num in zip(unknown_indices, perm):
            state_list[idx] = num
        state = tuple(state_list)
        if is_solvable(state):
            possible.append(state)
    return possible

class ComplexProblem:
    def __init__(self, initial_belief, goal_test, actions_func, results_func):
        self.initial_belief = initial_belief
        self.goal_test = goal_test
        self.actions = actions_func
        self.results = results_func
    def is_goal(self, state):
        return self.goal_test(state)

def and_or_graph_search(problem, max_depth=30, log_func=None):
    def or_search(state, path, depth):
        if log_func:
            log_func(f"OR_SEARCH state={state}, depth={depth}")
        if problem.is_goal(state):
            if log_func: log_func(f"Goal! {state}")
            return []
        if state in path:
            if log_func: log_func(f"Cycle {state}")
            return None
        if depth <= 0:
            return None
        for action in problem.actions(state):
            result_states = problem.results(state, action)
            if log_func:
                log_func(f"  Action {action} -> {result_states}")
            plan = and_search(result_states, path + [state], depth - 1)
            if plan is not None:
                return [action, plan]
        return None

    def and_search(states, path, depth):
        if log_func:
            log_func(f"AND_SEARCH states={states}, depth={depth}")
        if not states:
            return {}
        plans = {}
        for s in states:
            plan_s = or_search(s, path, depth)
            if plan_s is None:
                return None
            plans[s] = plan_s
        return plans

    return and_search(problem.initial_belief, path, max_depth)

def format_plan(plan, indent=0):
    if plan is None:
        return "Failure"
    if isinstance(plan, dict):
        lines = ["AND:"]
        for state, subplan in plan.items():
            lines.append("  " * indent + f"State {state}:")
            lines.append(format_plan(subplan, indent + 1))
        return "\n".join(lines)
    elif isinstance(plan, list):
        if not plan:
            return "  " * indent + "GOAL"
        action = plan[0]
        subplan = plan[1]
        return "  " * indent + f"Action: {action}\n" + format_plan(subplan, indent)
    else:
        return "  " * indent + str(plan)

# --- GIAO DIỆN CHÍNH ---
class EightPuzzleGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("8-Puzzle Search Algorithms")
        self.root.geometry("1250x800")
        self.root.configure(bg="#f5f6f8")
        self.current_state = list(GOAL)
        self.path = []
        self.current_step_idx = 0
        self.is_running = False
        self.top_mode = tk.StringVar(value="Standard")
        self.complex_submode = tk.StringVar(value="Unknown Start")
        self.complex_initial_belief = []
        self.complex_goal_states = [GOAL]
        self.conditional_plan = None
        self.setup_ui()
        self.reset_puzzle()

    def setup_ui(self):
        top_frame = tk.Frame(self.root, bg="#f5f6f8", height=50)
        top_frame.pack(side=tk.TOP, fill=tk.X, padx=20, pady=10)

        title_label = tk.Label(top_frame, text="8-Puzzle Search Algorithms", font=("Arial", 16, "bold"), bg="#f5f6f8", fg="#1e293b")
        title_label.pack(side=tk.LEFT)

        # Chọn chế độ Standard / Complex (sẽ tự động thay đổi khi chọn nhóm 4)
        tk.Label(top_frame, text="Mode:", bg="#f5f6f8").pack(side=tk.LEFT, padx=(20,5))
        self.mode_menu = ttk.Combobox(top_frame, textvariable=self.top_mode, values=["Standard", "Complex"], state="readonly", width=10)
        self.mode_menu.pack(side=tk.LEFT, padx=5)
        self.mode_menu.bind("<<ComboboxSelected>>", self.on_top_mode_change)

        # === STANDARD FRAME ===
        self.standard_frame = tk.Frame(top_frame, bg="#f5f6f8")
        self.standard_frame.pack(side=tk.LEFT, fill=tk.X, expand=True)

        # Nhóm thuật toán
        tk.Label(self.standard_frame, text="Group", bg="#f5f6f8").pack(side=tk.LEFT, padx=5)
        self.group_var = tk.IntVar(value=1)
        self.group_menu = ttk.Combobox(self.standard_frame, textvariable=self.group_var, 
                                       values=[1,2,3,4], state="readonly", width=3)
        self.group_menu.pack(side=tk.LEFT, padx=5)
        self.group_menu.bind("<<ComboboxSelected>>", self.on_group_change)

        # Thuật toán cụ thể
        tk.Label(self.standard_frame, text="Algorithm", bg="#f5f6f8").pack(side=tk.LEFT, padx=5)
        self.algo_var = tk.StringVar(value="BFS")
        self.algo_menu = ttk.Combobox(self.standard_frame, textvariable=self.algo_var, 
                                      values=["BFS", "DFS", "UCS", "A* (Misplaced)"], 
                                      width=25, state="readonly")
        self.algo_menu.pack(side=tk.LEFT, padx=5)

        tk.Label(self.standard_frame, text="Param", bg="#f5f6f8").pack(side=tk.LEFT, padx=5)
        self.max_depth_entry = tk.Spinbox(self.standard_frame, from_=1, to=200, width=5)
        self.max_depth_entry.delete(0, "end")
        self.max_depth_entry.insert(0, "35")
        self.max_depth_entry.pack(side=tk.LEFT, padx=5)

        self.btn_random = tk.Button(self.standard_frame, text="Random", bg="#ffffff", bd=1, relief=tk.SOLID, command=self.shuffle_puzzle, padx=10)
        self.btn_random.pack(side=tk.LEFT, padx=5)
        self.btn_run = tk.Button(self.standard_frame, text="Run", bg="#ffffff", bd=1, relief=tk.SOLID, command=self.start_solve, padx=15)
        self.btn_run.pack(side=tk.LEFT, padx=5)

        # === COMPLEX TOP FRAME (ẩn ban đầu) ===
        self.complex_top_frame = tk.Frame(top_frame, bg="#f5f6f8")
        self.complex_top_frame.pack(side=tk.LEFT, fill=tk.X, expand=True)
        self.complex_top_frame.pack_forget()
        self.btn_run_complex = tk.Button(self.complex_top_frame, text="Run Complex", bg="#ffffff", bd=1, relief=tk.SOLID, command=self.start_complex_solve, padx=15)
        self.btn_run_complex.pack(side=tk.LEFT, padx=10)

        self.btn_reset = tk.Button(top_frame, text="Reset", bg="#ffffff", bd=1, relief=tk.SOLID, command=self.reset_puzzle, padx=10)
        self.btn_reset.pack(side=tk.RIGHT, padx=5)

        # --- Main Layout ---
        main_frame = tk.Frame(self.root, bg="#f5f6f8")
        main_frame.pack(fill=tk.BOTH, expand=True, padx=20, pady=5)

        # LEFT COLUMN
        left_column = tk.Frame(main_frame, bg="#f5f6f8")
        left_column.pack(side=tk.LEFT, fill=tk.Y, padx=(0, 20))
        self.grid_frame = tk.Frame(left_column, bg="#f5f6f8")
        self.grid_frame.pack(pady=10)
        self.cells = []
        for i in range(9):
            btn = tk.Button(self.grid_frame, text="", font=("Arial", 22, "bold"), width=4, height=2, bg="#2563eb", fg="white", bd=0, highlightthickness=0)
            btn.grid(row=i//3, column=i%3, padx=4, pady=4)
            self.cells.append(btn)

        nav_frame = tk.Frame(left_column, bg="#f5f6f8")
        nav_frame.pack(fill=tk.X, pady=5)
        self.btn_prev = tk.Button(nav_frame, text="Prev Step", font=("Arial", 9), command=self.prev_step, bg="#ffffff", bd=1, relief=tk.SOLID)
        self.btn_prev.pack(side=tk.LEFT, expand=True, fill=tk.X, padx=2)
        self.btn_next = tk.Button(nav_frame, text="Next Step", font=("Arial", 9), command=self.next_step, bg="#ffffff", bd=1, relief=tk.SOLID)
        self.btn_next.pack(side=tk.LEFT, expand=True, fill=tk.X, padx=2)
        self.btn_auto = tk.Button(nav_frame, text="Auto Run", font=("Arial", 9), command=self.toggle_auto, bg="#ffffff", bd=1, relief=tk.SOLID)
        self.btn_auto.pack(side=tk.LEFT, expand=True, fill=tk.X, padx=2)

        speed_frame = tk.Frame(left_column, bg="#f5f6f8")
        speed_frame.pack(fill=tk.X, pady=5)
        tk.Label(speed_frame, text="Auto speed", bg="#f5f6f8", font=("Arial", 9)).pack(side=tk.LEFT)
        self.speed_scale = tk.Scale(speed_frame, from_=100, to=1000, orient=tk.HORIZONTAL, bg="#f5f6f8", bd=0, highlightthickness=0)
        self.speed_scale.set(300)
        self.speed_scale.pack(side=tk.RIGHT, fill=tk.X, expand=True, padx=(5, 0))

        goal_box = tk.LabelFrame(left_column, text="Goal", bg="#ffffff", font=("Arial", 9))
        goal_box.pack(fill=tk.BOTH, expand=True, pady=(15, 5))
        goal_matrix_lbl = tk.Label(goal_box, text="1  2  3\n4  5  6\n7  8  _", font=("Courier", 12, "bold"), bg="#ffffff", justify=tk.LEFT, pady=15)
        goal_matrix_lbl.pack()

        # Complex panel (ẩn/hiện theo mode)
        self.complex_panel = tk.LabelFrame(left_column, text="Complex Space Settings", bg="#f5f6f8", font=("Arial", 9))
        self.complex_panel.pack(fill=tk.BOTH, pady=5)
        self.complex_panel.pack_forget()
        self.setup_complex_panel()

        # RIGHT COLUMN: Info & Notebook
        right_column = tk.Frame(main_frame, bg="#f5f6f8")
        right_column.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        info_frame = tk.Frame(right_column, bg="#f5f6f8")
        info_frame.pack(fill=tk.X, pady=(0, 15))
        self.stats = {
            "Step": tk.StringVar(value="-"),
            "Algorithm": tk.StringVar(value="-"),
            "Expanded": tk.StringVar(value="-"),
            "Action": tk.StringVar(value="-"),
            "Path cost": tk.StringVar(value="-"),
            "Depth": tk.StringVar(value="-"),
            "IDS limit": tk.StringVar(value="-"),
            "Frontier": tk.StringVar(value="-"),
            "Explored": tk.StringVar(value="-"),
            "Status": tk.StringVar(value="Ready.")
        }
        keys = list(self.stats.keys())
        for idx, key in enumerate(keys):
            f = tk.Frame(info_frame, bg="#f5f6f8")
            if key == "Status":
                f.grid(row=9, column=0, columnspan=2, sticky="w", pady=4)
                tk.Label(f, text=f"{key}:", font=("Arial", 10, "bold"), bg="#f5f6f8", width=12, anchor="w").pack(side=tk.LEFT)
                tk.Label(f, textvariable=self.stats[key], font=("Arial", 10, "italic"), bg="#f5f6f8", fg="#475569").pack(side=tk.LEFT)
            else:
                f.grid(row=idx, column=0, sticky="w", pady=2)
                tk.Label(f, text=f"{key}:", font=("Arial", 10), bg="#f5f6f8", width=15, anchor="w").pack(side=tk.LEFT)
                tk.Label(f, textvariable=self.stats[key], font=("Arial", 10, "bold"), bg="#f5f6f8").pack(side=tk.LEFT)

        self.notebook = ttk.Notebook(right_column)
        self.notebook.pack(fill=tk.BOTH, expand=True)
        self.tab_children = tk.Text(self.notebook, font=("Courier", 10), bg="white", bd=0)
        self.tab_frontier = tk.Text(self.notebook, font=("Courier", 10), bg="white", bd=0)
        self.tab_explored = tk.Text(self.notebook, font=("Courier", 10), bg="white", bd=0)
        self.tab_steplog = tk.Text(self.notebook, font=("Courier", 10), bg="white", bd=0)
        self.tab_conditional = tk.Text(self.notebook, font=("Courier", 10), bg="white", bd=0)
        self.notebook.add(self.tab_children, text="Children")
        self.notebook.add(self.tab_frontier, text="Frontier")
        self.notebook.add(self.tab_explored, text="Explored")
        self.notebook.add(self.tab_steplog, text="Step Log")
        self.notebook.add(self.tab_conditional, text="Conditional Plan")

        # Khởi tạo danh sách thuật toán cho nhóm 1
        self.update_algo_menu()

    def on_group_change(self, event=None):
        g = self.group_var.get()
        if g == 4:
            # Tự động chuyển sang Complex mode
            self.top_mode.set("Complex")
            self.on_top_mode_change()
        else:
            if self.top_mode.get() == "Complex":
                self.top_mode.set("Standard")
                self.on_top_mode_change()
            self.update_algo_menu()

    def update_algo_menu(self):
        g = self.group_var.get()
        if g == 1:
            algos = ["BFS", "DFS", "UCS", "A* (Misplaced)"]
        elif g == 2:
            algos = ["Greedy Best-First", "UCS", "A* (Misplaced)"]
        elif g == 3:
            algos = ["Simple Hill Climbing", "Stochastic Hill Climbing", "Random Restart Hill Climbing", "Local Beam Search"]
        else:
            algos = []
        self.algo_menu['values'] = algos
        if algos:
            self.algo_var.set(algos[0])
        else:
            self.algo_var.set("")
        # Ẩn/hiện nút Run của Standard nếu không có algo (nhóm 4 không dùng)
        if g == 4:
            self.btn_run.pack_forget()
        else:
            self.btn_run.pack(side=tk.LEFT, padx=5, after=self.btn_random)  # đảm bảo hiển thị

    def on_top_mode_change(self, event=None):
        if self.top_mode.get() == "Complex":
            self.standard_frame.pack_forget()
            self.complex_top_frame.pack(side=tk.LEFT, fill=tk.X, expand=True)
            self.complex_panel.pack(fill=tk.BOTH, pady=5, after=self.grid_frame)
            self.update_complex_ui()
        else:
            self.complex_top_frame.pack_forget()
            self.complex_panel.pack_forget()
            self.standard_frame.pack(side=tk.LEFT, fill=tk.X, expand=True)
            self.update_algo_menu()
            # Đồng bộ group_var nếu cần
            if self.group_var.get() == 4:
                self.group_var.set(1)

    def setup_complex_panel(self):
        ctrl = tk.Frame(self.complex_panel, bg="#f5f6f8")
        ctrl.pack(fill=tk.X, pady=2)
        tk.Label(ctrl, text="Sub-mode:", bg="#f5f6f8").pack(side=tk.LEFT, padx=5)
        submode_menu = ttk.Combobox(ctrl, textvariable=self.complex_submode,
                                    values=["Unknown Start", "Unknown Goal", "Unknown Both", "Partially Known"],
                                    state="readonly", width=20)
        submode_menu.pack(side=tk.LEFT, padx=5)
        submode_menu.bind("<<ComboboxSelected>>", self.update_complex_ui)
        self.complex_dynamic = tk.Frame(self.complex_panel, bg="#f5f6f8")
        self.complex_dynamic.pack(fill=tk.BOTH, expand=True, pady=5)
        self.complex_actions = tk.Frame(self.complex_panel, bg="#f5f6f8")
        self.complex_actions.pack(fill=tk.X, pady=2)
        self.update_complex_ui()

    def update_complex_ui(self, event=None):
        for w in self.complex_dynamic.winfo_children():
            w.destroy()
        for w in self.complex_actions.winfo_children():
            w.destroy()
        submode = self.complex_submode.get()
        if submode == "Unknown Start":
            tk.Label(self.complex_dynamic, text="Initial belief (start states):", bg="#f5f6f8").pack(anchor="w")
            self.belief_text = tk.Text(self.complex_dynamic, height=5, width=30, font=("Courier", 9))
            self.belief_text.pack(fill=tk.BOTH, expand=True)
            tk.Button(self.complex_actions, text="Generate Random Starts", command=self.generate_start_belief).pack(side=tk.LEFT, padx=5)
            tk.Label(self.complex_actions, text="Count:", bg="#f5f6f8").pack(side=tk.LEFT)
            self.start_count = tk.Spinbox(self.complex_actions, from_=1, to=10, width=3)
            self.start_count.delete(0, "end"); self.start_count.insert(0, "3")
            self.start_count.pack(side=tk.LEFT)
        elif submode == "Unknown Goal":
            tk.Label(self.complex_dynamic, text="Goal belief (goal states):", bg="#f5f6f8").pack(anchor="w")
            self.belief_text = tk.Text(self.complex_dynamic, height=5, width=30, font=("Courier", 9))
            self.belief_text.pack(fill=tk.BOTH, expand=True)
            tk.Button(self.complex_actions, text="Generate Random Goals", command=self.generate_goal_belief).pack(side=tk.LEFT, padx=5)
            tk.Label(self.complex_actions, text="Count:", bg="#f5f6f8").pack(side=tk.LEFT)
            self.goal_count = tk.Spinbox(self.complex_actions, from_=1, to=10, width=3)
            self.goal_count.delete(0, "end"); self.goal_count.insert(0, "3")
            self.goal_count.pack(side=tk.LEFT)
        elif submode == "Unknown Both":
            tk.Label(self.complex_dynamic, text="Initial belief:", bg="#f5f6f8").pack(anchor="w")
            self.start_belief_text = tk.Text(self.complex_dynamic, height=3, width=30, font=("Courier", 9))
            self.start_belief_text.pack(fill=tk.BOTH, expand=True)
            tk.Label(self.complex_dynamic, text="Goal belief:", bg="#f5f6f8").pack(anchor="w")
            self.goal_belief_text = tk.Text(self.complex_dynamic, height=3, width=30, font=("Courier", 9))
            self.goal_belief_text.pack(fill=tk.BOTH, expand=True)
            act = tk.Frame(self.complex_actions, bg="#f5f6f8")
            act.pack(fill=tk.X)
            tk.Button(act, text="Generate Random Starts", command=self.generate_start_belief).pack(side=tk.LEFT, padx=5)
            tk.Label(act, text="Count:", bg="#f5f6f8").pack(side=tk.LEFT)
            self.start_count = tk.Spinbox(act, from_=1, to=10, width=3)
            self.start_count.delete(0, "end"); self.start_count.insert(0, "2")
            self.start_count.pack(side=tk.LEFT)
            tk.Button(act, text="Generate Random Goals", command=self.generate_goal_belief).pack(side=tk.LEFT, padx=10)
            tk.Label(act, text="Count:", bg="#f5f6f8").pack(side=tk.LEFT)
            self.goal_count = tk.Spinbox(act, from_=1, to=10, width=3)
            self.goal_count.delete(0, "end"); self.goal_count.insert(0, "2")
            self.goal_count.pack(side=tk.LEFT)
        elif submode == "Partially Known":
            tk.Label(self.complex_dynamic, text="Initial partial template:", bg="#f5f6f8").pack(anchor="w")
            self.template_entry = tk.Entry(self.complex_dynamic, font=("Courier", 10), width=25)
            self.template_entry.insert(0, "? 2 3 4 5 6 7 8 ?")
            self.template_entry.pack(fill=tk.X, pady=2)
            tk.Label(self.complex_dynamic, text="Goal partial template (optional):", bg="#f5f6f8").pack(anchor="w")
            self.goal_template_entry = tk.Entry(self.complex_dynamic, font=("Courier", 10), width=25)
            self.goal_template_entry.pack(fill=tk.X, pady=2)
            self.partial_display = tk.Text(self.complex_dynamic, height=5, width=30, font=("Courier", 9))
            self.partial_display.pack(fill=tk.BOTH, expand=True)
            tk.Button(self.complex_actions, text="Generate Belief from Templates", command=self.generate_partial_belief).pack(side=tk.LEFT, padx=5)

    # --- Các hàm sinh belief (giữ nguyên) ---
    def generate_start_belief(self):
        try: n = int(self.start_count.get())
        except: n = 3
        states = generate_random_solvable_states(n)
        self.complex_initial_belief = states
        if hasattr(self, 'belief_text'):
            self.belief_text.delete('1.0', tk.END)
            for s in states: self.belief_text.insert(tk.END, f"{s}\n")
        elif hasattr(self, 'start_belief_text'):
            self.start_belief_text.delete('1.0', tk.END)
            for s in states: self.start_belief_text.insert(tk.END, f"{s}\n")
            self.complex_initial_belief = states

    def generate_goal_belief(self):
        try: n = int(self.goal_count.get())
        except: n = 3
        states = generate_random_solvable_states(n)
        self.complex_goal_states = states
        if hasattr(self, 'belief_text'):
            self.belief_text.delete('1.0', tk.END)
            for s in states: self.belief_text.insert(tk.END, f"{s}\n")
        elif hasattr(self, 'goal_belief_text'):
            self.goal_belief_text.delete('1.0', tk.END)
            for s in states: self.goal_belief_text.insert(tk.END, f"{s}\n")

    def generate_partial_belief(self):
        start_tmpl = self.template_entry.get()
        goal_tmpl = self.goal_template_entry.get()
        start_states = parse_template(start_tmpl)
        if not start_states: return
        goal_states = parse_template(goal_tmpl) if goal_tmpl.strip() else [GOAL]
        if not goal_states: goal_states = [GOAL]
        self.complex_initial_belief = start_states
        self.complex_goal_states = goal_states
        self.partial_display.delete('1.0', tk.END)
        self.partial_display.insert(tk.END, "Start states:\n")
        for s in start_states: self.partial_display.insert(tk.END, f"{s}\n")
        self.partial_display.insert(tk.END, "\nGoal states:\n")
        for g in goal_states: self.partial_display.insert(tk.END, f"{g}\n")

    def start_complex_solve(self):
        submode = self.complex_submode.get()
        if submode == "Unknown Start":
            initial_belief = self.complex_initial_belief
            goal_states = [GOAL]
        elif submode == "Unknown Goal":
            initial_belief = [tuple(self.current_state)]
            goal_states = self.complex_goal_states
        elif submode == "Unknown Both":
            initial_belief = self.complex_initial_belief
            goal_states = self.complex_goal_states
        elif submode == "Partially Known":
            initial_belief = self.complex_initial_belief
            goal_states = self.complex_goal_states
        else: return
        if not initial_belief or not goal_states:
            messagebox.showwarning("Error", "Please generate belief states first.")
            return
        def actions(state): return [a for _, a in get_neighbors(state)]
        def results(state, action):
            for nstate, act in get_neighbors(state):
                if act == action: return [nstate]
            return []
        problem = ComplexProblem(initial_belief, lambda s: s in goal_states, actions, results)
        self.tab_children.delete('1.0', tk.END)
        self.tab_frontier.delete('1.0', tk.END)
        self.tab_explored.delete('1.0', tk.END)
        self.tab_steplog.delete('1.0', tk.END)
        self.tab_conditional.delete('1.0', tk.END)
        self.update_stat("Status", "Running AND-OR Graph Search...")
        self.btn_run_complex.config(state=tk.DISABLED)
        threading.Thread(target=self._complex_solve_thread, args=(problem,), daemon=True).start()

    def _complex_solve_thread(self, problem):
        start_time = time.time()
        max_depth = 30
        log_lines = []
        def log(msg):
            log_lines.append(msg)
            self.write_to_tab(self.tab_steplog, msg)
        plan = and_or_graph_search(problem, max_depth=max_depth, log_func=log)
        elapsed = time.time() - start_time
        if plan:
            self.conditional_plan = plan
            self.tab_conditional.insert('1.0', format_plan(plan))
            self.update_stat("Status", f"Found plan in {elapsed:.4f}s")
        else:
            self.tab_conditional.insert('1.0', "No plan found (failure).")
            self.update_stat("Status", "No plan found.")
        self.root.after(0, lambda: self.btn_run_complex.config(state=tk.NORMAL))

    # --- Các phương thức cũ (cập nhật giao diện, v.v.) ---
    def update_grid(self):
        for i, val in enumerate(self.current_state):
            if val == 0:
                self.cells[i].config(text="", bg="#e2e8f0")
            else:
                self.cells[i].config(text=str(val), bg="#2563eb")

    def reset_puzzle(self):
        self.current_state = list(GOAL)
        self.path = []
        self.current_step_idx = 0
        self.update_grid()
        self.tab_children.delete('1.0', tk.END)
        self.tab_frontier.delete('1.0', tk.END)
        self.tab_explored.delete('1.0', tk.END)
        self.tab_steplog.delete('1.0', tk.END)
        self.tab_conditional.delete('1.0', tk.END)
        for key, var in self.stats.items(): 
            var.set("-" if key != "Status" else "Ready.")
        Node._id_counter = 65

    def shuffle_puzzle(self):
        self.reset_puzzle()
        state = list(GOAL)
        for _ in range(14): 
            neighbors = get_neighbors(tuple(state))
            state = list(random.choice(neighbors)[0])
        self.current_state = state
        self.update_grid()

    def write_to_tab(self, tab, text):
        self.root.after(0, lambda: self._safe_write(tab, text))

    def _safe_write(self, tab, text):
        tab.insert(tk.END, text + "\n")
        tab.see(tk.END)

    def update_stat(self, key, value):
        self.root.after(0, lambda: self.stats[key].set(str(value)))

    def start_solve(self):
        if self.group_var.get() == 4:
            # Đã chuyển sang complex, không dùng start_solve
            return
        algo = self.algo_var.get()
        start_state = tuple(self.current_state)
        Node._id_counter = 65 
        try:
            param_val = int(self.max_depth_entry.get())
        except ValueError:
            param_val = 35
        self.tab_children.delete('1.0', tk.END)
        self.tab_frontier.delete('1.0', tk.END)
        self.tab_explored.delete('1.0', tk.END)
        self.tab_steplog.delete('1.0', tk.END)
        self.tab_conditional.delete('1.0', tk.END)
        self.btn_run.config(state=tk.DISABLED)
        threading.Thread(target=self.solve, args=(algo, start_state, param_val), daemon=True).start()

    def solve(self, algo, start, param_val):
        start_time = time.time()
        result_node = None
        explored = set()
        root_node = Node(start, cost=0, depth=0, h_cost=get_misplaced_tiles(start))

        # NHÓM 1 & 2 (các thuật toán có sẵn)
        if algo == "BFS":
            self.update_stat("Status", "Queue (FIFO)")
            queue = collections.deque([root_node])
            explored.add(start)
            while queue:
                node = queue.popleft()
                self.update_stat("Expanded", node.name)
                if node.state == GOAL:
                    result_node = node; break
                for state, action in get_neighbors(node.state):
                    if state not in explored and node.depth < param_val:
                        explored.add(state)
                        child = Node(state, node, action, node.cost + 1, node.depth + 1)
                        queue.append(child)
                        self.write_to_tab(self.tab_children, f"{child.name} | parent={node.name} | action={action} | depth={child.depth}\n{format_state_matrix(state)}")
                self.update_stat("Frontier", f"{len(queue)} node(s)")
                self.update_stat("Explored", f"{len(explored)} state(s)")

        elif algo == "DFS":
            self.update_stat("Status", "Stack (LIFO)")
            stack = [root_node]
            explored.add(start)
            while stack:
                node = stack.pop()
                self.update_stat("Expanded", node.name)
                if node.state == GOAL:
                    result_node = node; break
                if node.depth < param_val:
                    for state, action in get_neighbors(node.state):
                        if state not in explored:
                            explored.add(state)
                            child = Node(state, node, action, node.cost + 1, node.depth + 1)
                            stack.append(child)
                            self.write_to_tab(self.tab_children, f"{child.name} | parent={node.name} | action={action} | depth={child.depth}\n{format_state_matrix(state)}")
                self.update_stat("Frontier", f"{len(stack)} node(s)")
                self.update_stat("Explored", f"{len(explored)} state(s)")

        elif algo == "UCS":
            self.update_stat("Status", "Priority queue (cost)")
            pq = [(root_node.cost, id(root_node), root_node)]
            while pq:
                cost, _, node = heapq.heappop(pq)
                self.update_stat("Expanded", node.name)
                if node.state == GOAL:
                    result_node = node; break
                if node.state not in explored:
                    explored.add(node.state)
                    self.write_to_tab(self.tab_explored, f"Explored {node.name}:\n{format_state_matrix(node.state)}")
                    if node.depth < param_val:
                        for state, action in get_neighbors(node.state):
                            if state not in explored:
                                child = Node(state, node, action, node.cost + 1, node.depth + 1)
                                heapq.heappush(pq, (child.cost, id(child), child))
                                self.write_to_tab(self.tab_children, f"{child.name} | parent={node.name} | action={action} | cost={child.cost} | depth={child.depth}\n{format_state_matrix(state)}")
                self.update_stat("Frontier", f"{len(pq)} node(s)")
                self.update_stat("Explored", f"{len(explored)} state(s)")

        elif algo == "A* (Misplaced)":
            self.update_stat("Status", "A* (f = g + h)")
            pq = [(root_node.f_cost, id(root_node), root_node)]
            while pq:
                _, _, node = heapq.heappop(pq)
                self.update_stat("Expanded", node.name)
                if node.state == GOAL:
                    result_node = node; break
                if node.state not in explored:
                    explored.add(node.state)
                    self.write_to_tab(self.tab_explored, f"Explored {node.name}:\n{format_state_matrix(node.state)}")
                    if node.depth < param_val:
                        for state, action in get_neighbors(node.state):
                            if state not in explored:
                                h_val = get_misplaced_tiles(state)
                                child = Node(state, node, action, node.cost + 1, node.depth + 1, h_val)
                                heapq.heappush(pq, (child.f_cost, id(child), child))
                                self.write_to_tab(self.tab_children, f"{child.name} | parent={node.name} | action={action} | g={child.cost} h={child.h_cost} f={child.f_cost}\n{format_state_matrix(state)}")
                self.update_stat("Frontier", f"{len(pq)} node(s)")
                self.update_stat("Explored", f"{len(explored)} state(s)")

        # GREEDY BEST-FIRST (Nhóm 2 mới)
        elif algo == "Greedy Best-First":
            self.update_stat("Status", "Greedy (h only)")
            pq = [(root_node.h_cost, id(root_node), root_node)]
            explored_states = set()
            while pq:
                _, _, node = heapq.heappop(pq)
                self.update_stat("Expanded", node.name)
                if node.state == GOAL:
                    result_node = node; break
                if node.state not in explored_states:
                    explored_states.add(node.state)
                    self.write_to_tab(self.tab_explored, f"Explored {node.name}:\n{format_state_matrix(node.state)}")
                    if node.depth < param_val:
                        for state, action in get_neighbors(node.state):
                            if state not in explored_states:
                                h_val = get_misplaced_tiles(state)
                                child = Node(state, node, action, node.cost + 1, node.depth + 1, h_val)
                                heapq.heappush(pq, (child.h_cost, id(child), child))
                                self.write_to_tab(self.tab_children, f"{child.name} | parent={node.name} | action={action} | h={child.h_cost}\n{format_state_matrix(state)}")
                self.update_stat("Frontier", f"{len(pq)} node(s)")
                self.update_stat("Explored", f"{len(explored_states)} state(s)")

        # NHÓM 3: LOCAL SEARCH
        elif algo == "Simple Hill Climbing":
            self.update_stat("Status", "Simple Hill Climbing")
            current_node = root_node
            while True:
                self.update_stat("Expanded", current_node.name)
                self.write_to_tab(self.tab_explored, f"Current {current_node.name} (Value={get_value(current_node.state)})\n{format_state_matrix(current_node.state)}")
                if current_node.state == GOAL:
                    result_node = current_node; break
                next_node = None
                for state, action in get_neighbors(current_node.state):
                    if get_value(state) > get_value(current_node.state):
                        next_node = Node(state, current_node, action, current_node.cost + 1, current_node.depth + 1)
                        self.write_to_tab(self.tab_children, f"Found better {next_node.name} Value={get_value(state)}\n{format_state_matrix(state)}")
                        break
                if next_node:
                    current_node = next_node
                else:
                    self.write_to_tab(self.tab_steplog, f"Local maximum at {current_node.name}!")
                    result_node = current_node; break

        elif algo == "Stochastic Hill Climbing":
            self.update_stat("Status", "Stochastic Hill Climbing")
            current_node = root_node
            while True:
                self.update_stat("Expanded", current_node.name)
                self.write_to_tab(self.tab_explored, f"Current {current_node.name} (Value={get_value(current_node.state)})\n{format_state_matrix(current_node.state)}")
                if current_node.state == GOAL:
                    result_node = current_node; break
                better = []
                for state, action in get_neighbors(current_node.state):
                    if get_value(state) > get_value(current_node.state):
                        better.append((state, action))
                if not better:
                    self.write_to_tab(self.tab_steplog, f"Local maximum at {current_node.name}!")
                    result_node = current_node; break
                else:
                    state, action = random.choice(better)
                    current_node = Node(state, current_node, action, current_node.cost + 1, current_node.depth + 1)
                    self.write_to_tab(self.tab_children, f"Random chosen {current_node.name}\n{format_state_matrix(state)}")

        elif algo == "Random Restart Hill Climbing":
            self.update_stat("Status", "Random Restart Hill Climbing")
            max_restart = param_val
            self.write_to_tab(self.tab_steplog, f"Max restarts = {max_restart}")
            success = False
            for i in range(1, max_restart + 1):
                self.write_to_tab(self.tab_steplog, f"Restart {i}")
                if i == 1:
                    current_state_tuple = start
                else:
                    temp = list(GOAL)
                    for _ in range(18): temp = list(random.choice(get_neighbors(tuple(temp)))[0])
                    current_state_tuple = tuple(temp)
                current_node = Node(current_state_tuple, parent=None if i==1 else root_node, action=f"Restart {i}")
                while True:
                    if current_node.state == GOAL:
                        result_node = current_node; success = True; break
                    better = []
                    for state, action in get_neighbors(current_node.state):
                        if get_value(state) > get_value(current_node.state):
                            better.append((state, action))
                    if not better:
                        self.write_to_tab(self.tab_steplog, f"Local maximum at {current_node.name}")
                        break
                    better.sort(key=lambda x: get_value(x[0]), reverse=True)
                    best_state, best_action = better[0]
                    current_node = Node(best_state, current_node, best_action, current_node.cost + 1, current_node.depth + 1)
                if success: break
            if not success:
                self.write_to_tab(self.tab_steplog, "Failed all restarts!")
                result_node = current_node

        elif algo == "Local Beam Search":
            self.update_stat("Status", "Local Beam Search")
            k = param_val
            self.write_to_tab(self.tab_steplog, f"Beam width k = {k}")
            current_states = [root_node]
            step = 0
            while True:
                step += 1
                self.write_to_tab(self.tab_steplog, f"Step {step}")
                all_neighbors = []
                found_goal = None
                for node in current_states:
                    self.write_to_tab(self.tab_explored, f"Beam node {node.name}\n{format_state_matrix(node.state)}")
                    for state, action in get_neighbors(node.state):
                        child = Node(state, node, action, node.cost + 1, node.depth + 1)
                        all_neighbors.append(child)
                        if state == GOAL:
                            found_goal = child; break
                    if found_goal: break
                if found_goal:
                    result_node = found_goal; break
                if not all_neighbors:
                    self.write_to_tab(self.tab_steplog, "No neighbors, stop.")
                    result_node = current_states[0]; break
                all_neighbors.sort(key=lambda x: get_value(x.state), reverse=True)
                current_states = all_neighbors[:k]
                self.write_to_tab(self.tab_steplog, f"Selected top {len(current_states)}:")
                for n in current_states:
                    self.write_to_tab(self.tab_steplog, f"  {n.name} (Value={get_value(n.state)})")

        # Xử lý kết quả
        end_time = time.time()
        elapsed = end_time - start_time
        if result_node:
            self.path = []
            curr = result_node
            while curr:
                self.path.append(curr)
                curr = curr.parent
            self.path.reverse()
            self.current_step_idx = 0
            self.current_state = list(self.path[0].state)
            self.update_grid()
            self.update_stat("Algorithm", algo)
            self.update_stat("Path cost", result_node.cost)
            self.update_stat("Depth", result_node.depth)
            self.update_stat("Step", f"0 / {len(self.path)-1}")
            self.update_stat("Action", "Start Position")
            for idx, n in enumerate(self.path):
                act = n.action if n.action else "None"
                self.write_to_tab(self.tab_steplog, f"Step {idx}: Node {n.name} | Action: {act} | Value: {get_value(n.state)}")
            self.update_stat("Status", f"Found in {elapsed:.4f}s")
        else:
            self.update_stat("Status", "No path found!")
        self.root.after(0, lambda: self.btn_run.config(state=tk.NORMAL))

    # Prev/Next/Auto (giữ nguyên)
    def prev_step(self):
        if not self.path or self.current_step_idx <= 0: return
        self.current_step_idx -= 1
        node = self.path[self.current_step_idx]
        self.current_state = list(node.state)
        self.update_grid()
        self.update_stat("Step", f"{self.current_step_idx} / {len(self.path)-1}")
        self.update_stat("Action", node.action if node.action else "Start")
        self.update_stat("Path cost", node.cost)
        self.update_stat("Depth", node.depth)

    def next_step(self):
        if not self.path or self.current_step_idx >= len(self.path) - 1: return
        self.current_step_idx += 1
        node = self.path[self.current_step_idx]
        self.current_state = list(node.state)
        self.update_grid()
        self.update_stat("Step", f"{self.current_step_idx} / {len(self.path)-1}")
        self.update_stat("Action", node.action)
        self.update_stat("Path cost", node.cost)
        self.update_stat("Depth", node.depth)

    def toggle_auto(self):
        if self.is_running:
            self.is_running = False
            self.btn_auto.config(text="Auto Run", bg="#ffffff")
        else:
            if not self.path: return
            self.is_running = True
            self.btn_auto.config(text="Stop Auto", bg="#ef4444", fg="white")
            threading.Thread(target=self.auto_loop, daemon=True).start()

    def auto_loop(self):
        while self.is_running and self.current_step_idx < len(self.path) - 1:
            self.root.after(0, self.next_step)
            time.sleep(self.speed_scale.get() / 1000.0)
        self.is_running = False
        self.root.after(0, lambda: self.btn_auto.config(text="Auto Run", bg="#ffffff", fg="black"))

if __name__ == "__main__":
    root = tk.Tk()
    app = EightPuzzleGUI(root)
    root.mainloop()